# ISMN Data Exploration
Visualising soil moisture data across stations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

DATA_DIR = '../data/ismn_data'

In [ ]:
# Load a single station
def load_station(network, station):
    path = os.path.join(DATA_DIR, network, f'{station}.csv')
    df = pd.read_csv(path, parse_dates=['datetime_utc'])
    df = df[df['ismn_flag_good'] == True]  # only good quality readings
    df = df.sort_values('datetime_utc')
    return df

# List all available stations
stations = []
for network in os.listdir(DATA_DIR):
    net_path = os.path.join(DATA_DIR, network)
    if os.path.isdir(net_path):
        for f in os.listdir(net_path):
            if f.endswith('.csv'):
                stations.append((network, f[:-4]))

print(f'{len(stations)} stations available:')
for n, s in stations:
    print(f'  {n} / {s}')

In [ ]:
# Plot soil moisture over time for one station
network, station = 'TAHMO', 'Mahali_Mzuri'  # change this to any station

df = load_station(network, station)

plt.figure(figsize=(14, 4))
plt.plot(df['datetime_utc'], df['sm_value'], linewidth=0.8)
plt.axhline(0.30, color='red', linestyle='--', label='Water threshold (0.30)')
plt.title(f'Soil Moisture — {station}')
plt.xlabel('Date')
plt.ylabel('Soil Moisture (m³/m³)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compare average soil moisture across all stations
summary = []
for network, station in stations:
    df = load_station(network, station)
    if len(df) > 0:
        summary.append({
            'station': station,
            'network': network,
            'mean_sm': df['sm_value'].mean(),
            'n_readings': len(df)
        })

summary_df = pd.DataFrame(summary).sort_values('mean_sm')

plt.figure(figsize=(14, 8))
colors = ['steelblue' if n == 'TAHMO' else 'darkorange' for n in summary_df['network']]
plt.barh(summary_df['station'], summary_df['mean_sm'], color=colors)
plt.axvline(0.30, color='red', linestyle='--', label='Water threshold (0.30)')
plt.xlabel('Mean Soil Moisture (m³/m³)')
plt.title('Average Soil Moisture by Station')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Flag quality breakdown across all data
all_flags = []
for network, station in stations:
    path = os.path.join(DATA_DIR, network, f'{station}.csv')
    df = pd.read_csv(path)
    all_flags.append(df['ismn_flag_raw'].value_counts())

flag_counts = pd.concat(all_flags).groupby(level=0).sum().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
flag_counts.plot(kind='bar')
plt.title('Data Quality Flag Distribution (all stations)')
plt.xlabel('Flag')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(flag_counts)